# Study 811 — Zero-Return Illiquidity — the teardown

The signal degeneracy, the per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 3894, 'fingerprint': '357fd262912f', 'zp_min_pct': 0.0, 'zp_med_pct': 0.0, 'zp_max_pct': 2.38, 'long_zp_pct': 1.38, 'spread_bps': -1.37, 't_nw': -1.29, 't_1s': -1.29, 'hi_bps': 6.55, 'lo_bps': 7.93, 'welch_t': -0.52, 'gross_sharpe': -0.33, 'placebo_obs': -1.37, 'placebo_mean': 0.123, 'placebo_sd': 1.108, 'placebo_p': 0.907, 'placebo_sigma_left': 1.35, 'placebo_draws': 1000, 'era_early_bps': -2.23, 'era_early_t': -1.69, 'era_early_n': 1760, 'era_late_bps': -0.66, 'era_late_t': -0.41, 'era_late_n': 2134, 'timer_1_gross': -1.37, 'timer_1_cost': 2.14, 'timer_1_net': -3.51, 'timer_1_t': -3.3, 'timer_5_gross': -1.37, 'timer_5_cost': 10.14, 'timer_5_net': -11.51, 'timer_5_t': -10.83, 'null_mean_t': -0.67, 'null_sd_t': 0.88, 'null_fire': 0, 'planted_t': 11.23, 'planted_welch': 10.98}

## Signal degeneracy — the reason to expect None

Trailing-252-day zero-return proportion across the 50 names (final row).

In [2]:
print(f"zero-return proportion: min {R['zp_min_pct']:.2f}%  median {R['zp_med_pct']:.2f}%  "
      f"max {R['zp_max_pct']:.2f}%  of days")
print(f"long (top-30%) book carries only ~{R['long_zp_pct']:.2f}% zero-days on average")
print('half the mega-caps sit at exactly 0.00% -> the short leg is a tie-broken pile')

zero-return proportion: min 0.00%  median 0.00%  max 2.38%  of days
long (top-30%) book carries only ~1.38% zero-days on average
half the mega-caps sit at exactly 0.00% -> the short leg is a tie-broken pile


## The headline — long-high-zero / short-low-zero spread

Daily equal-weight top-30% minus bottom-30% zero-proportion spread.

In [3]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : illiquid {R['hi_bps']:+.2f} vs liquid {R['lo_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : -1.37 bps/day  NW(10) t = -1.29  one-sample t = -1.29
books         : illiquid +6.55 vs liquid +7.93 bps (Welch t = -0.52)
gross Sharpe  : -0.33 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

In [4]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> right-tail p = {R['placebo_p']:.3f} "
      f"(~{R['placebo_sigma_left']:.1f} sigma into the left tail)")

observed -1.37 bps vs placebo mean +0.123 (sd 1.108) -> right-tail p = 0.907 (~1.4 sigma into the left tail)


## Robustness — two eras (split 2018-01-01)

In [5]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")
print('neither era clears |t| >= 2')

2010-2017 (n=1760): -2.23 bps  NW t = -1.69
2018-2026 (n=2134): -0.66 bps  NW t = -0.41
neither era clears |t| >= 2


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow. (And the long leg is the least-liquid names — the charged cost is a floor.)

In [6]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross -1.37 -> net -3.51 bps/day (cost 2.14/day, t=-3.30)
5 bps one-way: gross -1.37 -> net -11.51 bps/day (cost 10.14/day, t=-10.83)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted premium.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from zero_return import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=811+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.012, seed=811, n_assets=40, n_days=1500))
print(f"planted (edge=0.012): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.60 (sd 0.76), |t|>=2 in 0/8


planted (edge=0.012): NW t = +11.23, Welch t = +10.98


## Verdict

- **Signal — None.** The Lesmond-Ogden-Trzcinka zero-return illiquidity premium does **not** appear on 50 liquid US mega-caps: the long-high-zero / short-low-zero spread is **-1.37 bps/day** (NW *t* = **-1.29**, |t| < 2), insignificant and weakly wrong-signed, not stable across eras (*t* = -1.69 / -0.41). The proxy is near-degenerate here (median trailing zero-proportion 0.00%) — the effect lives in small, tick-priced names, not mega-caps. The 20-seed synthetic control recovers a *planted* premium cleanly (*t* = +11.23, fires on 0/20 nulls), so the flat real result is a true absence, not a broken sort.
- **Tradability — Mirage.** The book loses money gross (-1.37 bps/day) and worse net (**-3.51 bps/day** at 1 bp, *t* = -3.30; **-11.51** at 5 bps), and the long leg's real costs are far above the optimistic floor charged.